## Testing twitter model

In [ ]:
import os
import numpy as np
from params.paths import DATA_DIR
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# -----------------------------
# CONFIG (match training)
# -----------------------------
MAX_LEN = 256
LABELS = ["政治", "その他"]


# Option B: if you pushed the FULL model to Hugging Face, use:
MODEL_DIR = "kkatodus/text_labeller_twitter-full-deberta-v3-base-japanese"

device = "cuda" if torch.cuda.is_available() else "cpu"

# -----------------------------
# LOAD (full model, no adapters)
# -----------------------------
cfg = AutoConfig.from_pretrained(MODEL_DIR)

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=False)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(device).eval()

# Make sure config has correct label mapping (in case it didn't get saved)
model.config.id2label = {i: lab for i, lab in enumerate(LABELS)}
model.config.label2id = {lab: i for i, lab in enumerate(LABELS)}
model.config.num_labels = 2

print("Loaded model from:", MODEL_DIR)
print("num_labels:", model.config.num_labels)
print("id2label:", model.config.id2label)

# -----------------------------
# PREDICT
# -----------------------------
@torch.no_grad()
def predict(texts: list[str], topk: int = 2):
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LEN,
        padding=True,
    ).to(device)

    logits = model(**inputs).logits              # (B, 2)
    probs = torch.softmax(logits, dim=-1)        # (B, 2)

    probs_np = probs.cpu().numpy()
    preds = probs.argmax(dim=-1).cpu().numpy()

    out = []
    for i in range(len(texts)):
        pred_id = int(preds[i])
        row = probs_np[i]
        top_ids = row.argsort()[-topk:][::-1]
        out.append({
            "pred": model.config.id2label[pred_id],
            "confidence": float(row[pred_id]),
            "topk": [(model.config.id2label[int(j)], float(row[int(j)])) for j in top_ids],
            "probs": row.tolist(),  # [p(政治), p(その他)] in label-id order
        })
    return out

# -----------------------------
# OPTIONAL: evaluate on labeled set
# labels must be 0/1 ints (same as training)
# -----------------------------
@torch.no_grad()
def evaluate_texts(texts: list[str], true_labels: list[int]):
    preds_info = predict(texts)
    pred_ids = np.array([0 if p["pred"] == "政治" else 1 for p in preds_info], dtype=np.int64)
    y = np.array(true_labels, dtype=np.int64)

    return {
        "accuracy": accuracy_score(y, pred_ids),
        "f1_macro": f1_score(y, pred_ids, average="macro", zero_division=0),
        "f1_micro": f1_score(y, pred_ids, average="micro", zero_division=0),
        "precision_macro": precision_score(y, pred_ids, average="macro", zero_division=0),
        "recall_macro": recall_score(y, pred_i ds, average="macro", zero_division=0),
    }

# -----------------------------
# DEMO
# -----------------------------
prediction_strs = [
    "今日は国会で防衛費について議論がありました。",
    "森友問題にまた新たな疑惑です。安倍昭恵夫人付の秘書だった谷査恵氏が財務省に問い合わせをした際のやり取りの内容や…",
    "朝早くから40人ほど集まって、にぎやかに宣伝を行いました。",
    "博多の屋台でラーメン食べたいなあ。",
    "新しいスマホを買いました！すごく使いやすいです。",
]

preds = predict(prediction_strs)

for t, p in zip(prediction_strs, preds):
    print(t)
    print("pred:", p["pred"], "conf:", p["confidence"])
    print("topk:", p["topk"])
    print("probs:", p["probs"])
    print()


Loaded model from: kkatodus/text_labeller_twitter-full-deberta-v3-base-japanese
num_labels: 2
id2label: {0: '政治', 1: 'その他'}
今日は国会で防衛費について議論がありました。
pred: 政治 conf: 0.9996389150619507
topk: [('政治', 0.9996389150619507), ('その他', 0.0003610982676036656)]
probs: [0.9996389150619507, 0.0003610982676036656]

森友問題にまた新たな疑惑です。安倍昭恵夫人付の秘書だった谷査恵氏が財務省に問い合わせをした際のやり取りの内容や…
pred: 政治 conf: 0.9996328353881836
topk: [('政治', 0.9996328353881836), ('その他', 0.0003671098966151476)]
probs: [0.9996328353881836, 0.0003671098966151476]

朝早くから40人ほど集まって、にぎやかに宣伝を行いました。
pred: その他 conf: 0.9988155364990234
topk: [('その他', 0.9988155364990234), ('政治', 0.0011845327680930495)]
probs: [0.0011845327680930495, 0.9988155364990234]

博多の屋台でラーメン食べたいなあ。
pred: その他 conf: 0.9996854066848755
topk: [('その他', 0.9996854066848755), ('政治', 0.00031457559089176357)]
probs: [0.00031457559089176357, 0.9996854066848755]

新しいスマホを買いました！すごく使いやすいです。
pred: その他 conf: 0.9996742010116577
topk: [('その他', 0.9996742010116577), ('政治', 0.00032581144478172064)]
prob

## Testing parliament model (multi-label)

This model predicts **0+ labels per text** (sigmoid per label + threshold), unlike the Twitter model above which is **single-label softmax**.

In [4]:
import numpy as np
import torch
from transformers import AutoConfig, AutoTokenizer, AutoModelForSequenceClassification

# -----------------------------
# CONFIG (match training)
# -----------------------------
LABELS = [
    "質問文",
    "追及・確認文",
    "答弁文",
    "反論・再反論文",
    "意見文",
    "要望・提案文",
    "批判文",
    "評価文",
    "説明文",
    "事実文",
    "謝罪・釈明文",
    "手続・運営文",
    "その他",
]

MODEL_DIR = "kkatodus/text_labeller_parliament-deberta-v3-base-japanese"
MAX_LEN = 512
THRESHOLD = 0.5
TOPK = 5

device = "cuda" if torch.cuda.is_available() else "cpu"

# Build an explicit config so the classifier head is created with 13 labels
cfg = AutoConfig.from_pretrained(MODEL_DIR)
cfg.id2label = {i: lab for i, lab in enumerate(LABELS)}
cfg.label2id = {lab: i for i, lab in enumerate(LABELS)}
cfg.num_labels = len(LABELS)
cfg.problem_type = "multi_label_classification"

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=False)

print(cfg.num_labels)

# -----------------------------
# LOAD (adapter repo or merged/full)
# -----------------------------
try:
    from peft import AutoPeftModelForSequenceClassification

    model = AutoPeftModelForSequenceClassification.from_pretrained(MODEL_DIR, config=cfg)
except Exception:
    # For merged/full repos, this is the normal path.
    # `ignore_mismatched_sizes=True` makes this robust if the remote config is stale.
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_DIR,
        config=cfg,
        ignore_mismatched_sizes=True,
    )

model = model.to(device).eval()

print("Loaded model from:", MODEL_DIR)
print("num_labels:", model.config.num_labels)


def sigmoid(x: torch.Tensor) -> torch.Tensor:
    return 1 / (1 + torch.exp(-x))


@torch.no_grad()
def predict_parliament(texts: list[str], threshold: float = THRESHOLD, topk: int = TOPK):
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LEN,
        padding=True,
    ).to(device)

    logits = model(**inputs).logits  # (B, 13)
    probs = sigmoid(logits).cpu().numpy()

    out = []
    for i, t in enumerate(texts):
        row = probs[i]
        chosen = [model.config.id2label[j] for j, p in enumerate(row) if float(p) >= threshold]
        top_ids = row.argsort()[-topk:][::-1]
        top = [(model.config.id2label[int(j)], float(row[int(j)])) for j in top_ids]
        out.append({
            "labels(thr)": chosen,
            "topk": top,
            "probs": row.tolist(),
        })
    return out


# -----------------------------
# DEMO
# -----------------------------
prediction_strs = [
	"ハンバーガが大好きです。",
    "ただいま議題となっております法案について、政府の見解を伺います。",
    "ご指摘の点については、事実関係を確認の上、適切に対応いたします。",
    "そのような批判は当たりません。これまでの経緯を踏まえれば明らかです。",
    "私はこの政策を高く評価いたします。",
	"現場の先生には多くの負担をかけたのかもしれません"
	"しかし、公立の学校では相当数が出てきて、結果、国立ではそういう問題はなかったですよね",
	"二〇〇一年九月十一日のアメリカ同時多発テロの事件から四年がたちました",
	'今回のこの附属学校の問題ででも今明らかにさせていただきましたが、なぜ国立大学の附属学校がこれだけ進学校化してしまったのか',
	"お二人以外に、航空ネットワーク部長、セメントなどの建設資材の販売などを営む会社経営者と会社関係者二名、計六名の会食だったと聞いています"
]

preds = predict_parliament(prediction_strs, threshold=0.55, topk=3)
for t, p in zip(prediction_strs, preds):
    print(t)
    print("labels(thr):", p["labels(thr)"])
    print("topk:", p["topk"])
    print()

13


Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at ku-nlp/deberta-v3-base-japanese and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/root/projects/kokkai_analysis/data/data_venv/lib/python3.14/site-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at ku-nlp/deb

Loaded model from: kkatodus/text_labeller_parliament-deberta-v3-base-japanese
num_labels: 13
ハンバーガが大好きです。
labels(thr): []
topk: [('批判文', 0.5472885966300964), ('意見文', 0.5072885751724243), ('説明文', 0.4829409420490265)]

ただいま議題となっております法案について、政府の見解を伺います。
labels(thr): []
topk: [('意見文', 0.5358930826187134), ('批判文', 0.5129807591438293), ('答弁文', 0.5043995976448059)]

ご指摘の点については、事実関係を確認の上、適切に対応いたします。
labels(thr): ['批判文']
topk: [('批判文', 0.5595163106918335), ('意見文', 0.542994499206543), ('事実文', 0.49997007846832275)]

そのような批判は当たりません。これまでの経緯を踏まえれば明らかです。
labels(thr): []
topk: [('批判文', 0.5457817316055298), ('意見文', 0.5337276458740234), ('要望・提案文', 0.5103377103805542)]

私はこの政策を高く評価いたします。
labels(thr): ['批判文']
topk: [('批判文', 0.5774976015090942), ('意見文', 0.5230712294578552), ('事実文', 0.48395630717277527)]

現場の先生には多くの負担をかけたのかもしれませんしかし、公立の学校では相当数が出てきて、結果、国立ではそういう問題はなかったですよね
labels(thr): ['批判文']
topk: [('批判文', 0.5558557510375977), ('意見文', 0.539783239364624), ('手続・運営文', 0.5064747333526611)]

二〇〇一年九月十一日のアメリカ同時多発テロ

## Testing Youtube model

In [12]:
import numpy as np
import torch
from transformers import AutoConfig, AutoTokenizer, AutoModelForSequenceClassification

# -----------------------------
# CONFIG (match training)
# -----------------------------
LABELS = [
    "政治",
	"その他",
	"質問文",
	"追及・確認文",
	"答弁文",
	"反論・再反論文",
	"意見文",
	"要望・提案文",
	"批判文",
	"評価文",
	"説明文",
	"事実文",
	"謝罪・釈明文",
	"手続・運営文",
]

MODEL_DIR = "kkatodus/text_labeller_youtube-deberta-v3-base-japanese"
MAX_LEN = 512
THRESHOLD = 0.5
TOPK = 5

device = "cuda" if torch.cuda.is_available() else "cpu"

# Build an explicit config so the classifier head is created with 13 labels
cfg = AutoConfig.from_pretrained(MODEL_DIR)
cfg.id2label = {i: lab for i, lab in enumerate(LABELS)}
cfg.label2id = {lab: i for i, lab in enumerate(LABELS)}
cfg.num_labels = len(LABELS)
cfg.problem_type = "multi_label_classification"

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=False)

print(cfg.num_labels)

# -----------------------------
# LOAD (adapter repo or merged/full)
# -----------------------------
try:
    from peft import AutoPeftModelForSequenceClassification

    model = AutoPeftModelForSequenceClassification.from_pretrained(MODEL_DIR, config=cfg)
except Exception:
    # For merged/full repos, this is the normal path.
    # `ignore_mismatched_sizes=True` makes this robust if the remote config is stale.
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_DIR,
        config=cfg,
        ignore_mismatched_sizes=True,
    )

model = model.to(device).eval()

print("Loaded model from:", MODEL_DIR)
print("num_labels:", model.config.num_labels)


def sigmoid(x: torch.Tensor) -> torch.Tensor:
    return 1 / (1 + torch.exp(-x))


@torch.no_grad()
def predict_parliament(texts: list[str], threshold: float = THRESHOLD, topk: int = TOPK):
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LEN,
        padding=True,
    ).to(device)

    logits = model(**inputs).logits  # (B, 13)
    probs = sigmoid(logits).cpu().numpy()

    out = []
    for i, t in enumerate(texts):
        row = probs[i]
        chosen = [model.config.id2label[j] for j, p in enumerate(row) if float(p) >= threshold]
        top_ids = row.argsort()[-topk:][::-1]
        top = [(model.config.id2label[int(j)], float(row[int(j)])) for j in top_ids]
        out.append({
            "labels(thr)": chosen,
            "topk": top,
            "probs": row.tolist(),
        })
    return out


# -----------------------------
# DEMO
# -----------------------------
prediction_strs = [
    "皆さんこんにちは、岡田勝也です。今日はちょっと違う話で、私のおやつの話をしたいと思います。私が甘党だ。ということはご存知の方も多いかと思います。が、酒は全く飲みませんし、甘いものには芽がない。ただ。し、ケーキなどはカロリーが多いです。から、最近はなるべく我慢するようにしています。私の楽しみは羊羹を食べることです。昼の食事はこれもまたいつか申し上げたと思います。が特別のジュースを飲んでそれで昼食に帰るという生活を民主党代表時代からです。からずいぶん長く続けています。もちろん昼食会など入るときは別です。がそれ以外は粉を溶いてジュースを作ってもらってそれを5秒くらいで飲んでしまって終わりということです。カロリーは少なく栄養のバランスの取れたジュースなんです。がただ。どうしても夕方になると少しお腹が空くとそういう時に楽しみにしているのが羊羹です。ほぼ毎日ちょっと薄めに切ってもらいます。が2切れ食べるということを続けていまして東京で買った羊羹もあります。しあるいは週末にそれぞれ対話集会で行った時に空港でそのうちの羊羹を買うということもあります。この前川川に行った時はナルトのナルト金時の羊羹をこれ買ったのではなくて実はもらったんです。がそれを食べたりして楽しんでいます。8日も結構軽い高いので気をつけなければと思いながら疲労回復には甘いものが非常にいいということを改めて実感しています。今日は以上です。",
    "ご指摘の点については、事実関係を確認の上、適切に対応いたします。",
    "そのような批判は当たりません。これまでの経緯を踏まえれば明らかです。",
    "私はこの政策を高く評価いたします。",
]

preds = predict_parliament(prediction_strs)
for t, p in zip(prediction_strs, preds):
    print(t)
    print("labels(thr):", p["labels(thr)"])
    print("topk:", p["topk"])
    print()

14


Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at ku-nlp/deberta-v3-base-japanese and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at ku-nlp/deberta-v3-base-japanese and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loaded model from: kkatodus/text_labeller_youtube-deberta-v3-base-japanese
num_labels: 14
皆さんこんにちは、岡田勝也です。今日はちょっと違う話で、私のおやつの話をしたいと思います。私が甘党だ。ということはご存知の方も多いかと思います。が、酒は全く飲みませんし、甘いものには芽がない。ただ。し、ケーキなどはカロリーが多いです。から、最近はなるべく我慢するようにしています。私の楽しみは羊羹を食べることです。昼の食事はこれもまたいつか申し上げたと思います。が特別のジュースを飲んでそれで昼食に帰るという生活を民主党代表時代からです。からずいぶん長く続けています。もちろん昼食会など入るときは別です。がそれ以外は粉を溶いてジュースを作ってもらってそれを5秒くらいで飲んでしまって終わりということです。カロリーは少なく栄養のバランスの取れたジュースなんです。がただ。どうしても夕方になると少しお腹が空くとそういう時に楽しみにしているのが羊羹です。ほぼ毎日ちょっと薄めに切ってもらいます。が2切れ食べるということを続けていまして東京で買った羊羹もあります。しあるいは週末にそれぞれ対話集会で行った時に空港でそのうちの羊羹を買うということもあります。この前川川に行った時はナルトのナルト金時の羊羹をこれ買ったのではなくて実はもらったんです。がそれを食べたりして楽しんでいます。8日も結構軽い高いので気をつけなければと思いながら疲労回復には甘いものが非常にいいということを改めて実感しています。今日は以上です。
labels(thr): ['政治', '意見文', '評価文', '説明文']
topk: [('説明文', 0.5713438987731934), ('政治', 0.5567315220832825), ('評価文', 0.5204182863235474), ('意見文', 0.5142884254455566), ('要望・提案文', 0.49319326877593994)]

ご指摘の点については、事実関係を確認の上、適切に対応いたします。
labels(thr): ['政治', '意見文', '評価文', '説明文']
topk: [('説明文', 0.5777037739753723),